# UMAP: Uniform Manifold Approximation and Projection

Notebook นี้ใช้ UMAP เพื่อสร้าง embedding 2 มิติจาก Digits dataset และประเมินว่า embedding รักษาเพื่อนบ้านเดิมได้ดีเพียงใด

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Load and Prepare Digits Data

Digits มีตัวอย่างภาพตัวเลข $8 \times 8$ จำนวน 1,797 ภาพ แต่ละภาพแทนด้วย 64 pixel-intensity features. UMAP อาศัยระยะทาง จึง standardize ก่อนสร้าง embedding.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Original shape:', X.shape)
print('Standardized shape:', X_scaled.shape)
pd.Series(y, name='digit').value_counts().sort_index().to_frame('count')

## 2. PCA versus UMAP

PCA เป็น linear projection ที่รักษา variance ส่วน UMAP สร้างกราฟของเพื่อนบ้านในพื้นที่เดิม แล้วจัดวางกราฟนั้นใน 2D. สีใช้ label เพื่อช่วยตรวจสอบภายหลังเท่านั้น ไม่ได้ส่งให้ UMAP ระหว่าง fit.

In [ ]:
pca_embedding = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
umap_embedding = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=RANDOM_STATE,
).fit_transform(X_scaled)

figure, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, embedding, title in [
    (axes[0], pca_embedding, 'PCA projection'),
    (axes[1], umap_embedding, 'UMAP embedding'),
]:
    scatter = axis.scatter(embedding[:, 0], embedding[:, 1], c=y, cmap='tab10', s=12, alpha=0.8)
    axis.set(title=title, xlabel='Component 1', ylabel='Component 2')
    axis.grid(alpha=0.2)
figure.colorbar(scatter, ax=axes, ticks=range(10), label='Digit label')
plt.tight_layout()
plt.show()

## 3. Effect of `n_neighbors`

`n_neighbors` กำหนดขนาดของ neighborhood graph: ค่าน้อยเน้นโครงสร้างเฉพาะที่ ส่วนค่ามากนำบริบทที่กว้างขึ้นมาใช้. เปลี่ยนทีละ parameter เพื่อให้ตีความผลได้.

In [ ]:
neighbor_counts = [5, 15, 50, 100]
figure, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, neighbor_count in zip(axes.flat, neighbor_counts):
    embedding = umap.UMAP(
        n_neighbors=neighbor_count,
        min_dist=0.1,
        random_state=RANDOM_STATE,
    ).fit_transform(X_scaled)
    axis.scatter(embedding[:, 0], embedding[:, 1], c=y, cmap='tab10', s=10, alpha=0.8)
    axis.set(title=f'n_neighbors = {neighbor_count}')
    axis.set_xticks([])
    axis.set_yticks([])

plt.tight_layout()
plt.show()

## 4. Effect of `min_dist`

`min_dist` ควบคุมความใกล้กันต่ำสุดที่ UMAP อนุญาตใน embedding: ค่าน้อยทำให้กลุ่มดูแน่นขึ้น แต่ไม่ได้เป็นหลักฐานว่า groups แยกกันจริงในข้อมูลต้นฉบับ.

In [ ]:
minimum_distances = [0.0, 0.1, 0.3, 0.7]
figure, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, minimum_distance in zip(axes.flat, minimum_distances):
    embedding = umap.UMAP(
        n_neighbors=15,
        min_dist=minimum_distance,
        random_state=RANDOM_STATE,
    ).fit_transform(X_scaled)
    axis.scatter(embedding[:, 0], embedding[:, 1], c=y, cmap='tab10', s=10, alpha=0.8)
    axis.set(title=f'min_dist = {minimum_distance}')
    axis.set_xticks([])
    axis.set_yticks([])

plt.tight_layout()
plt.show()

## 5. Evaluate Neighborhood Preservation

Trustworthiness วัดว่าเพื่อนบ้านใน embedding 2D เคยเป็นเพื่อนบ้านในพื้นที่เดิมหรือไม่. ค่าสูงใกล้ 1 หมายถึงการรักษา local neighborhoods ได้ดีขึ้น.

In [ ]:
evaluation_neighbors = [5, 10, 15, 30, 50, 100]
evaluation_rows = []

for neighbor_count in evaluation_neighbors:
    embedding = umap.UMAP(
        n_neighbors=neighbor_count,
        min_dist=0.1,
        random_state=RANDOM_STATE,
    ).fit_transform(X_scaled)
    evaluation_rows.append({
        'n_neighbors': neighbor_count,
        'trustworthiness': trustworthiness(X_scaled, embedding, n_neighbors=10),
    })

trustworthiness_results = pd.DataFrame(evaluation_rows)
trustworthiness_results.round(3)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(
    trustworthiness_results['n_neighbors'],
    trustworthiness_results['trustworthiness'],
    marker='o',
    color='#2563eb',
)
plt.ylim(0.75, 1.01)
plt.xlabel('UMAP n_neighbors')
plt.ylabel('Trustworthiness')
plt.title('Local-neighborhood preservation')
plt.grid(alpha=0.25)
plt.show()

## Interpretation Checklist

- การแยกกลุ่มในภาพไม่ได้พิสูจน์ว่าคลาสแยกได้จริงเสมอไป
- เปลี่ยนทีละ parameter และตรวจว่า local relationships ที่สำคัญยังคงอยู่
- ใช้ trustworthiness ประกอบกับ domain knowledge และ metric ของงานจริง
- ตั้ง `random_state` สำหรับการสอนหรือเปรียบเทียบที่ต้องการทำซ้ำได้
- สำหรับโมเดลทำนาย ให้ประเมินด้วย train/test split หรือ cross-validation ไม่ใช่ดู embedding เพียงอย่างเดียว